In [151]:
import pandas as pd 
from datetime import datetime

today_str = datetime.today().strftime("%d_%m_%Y")
raw_filename = f"data_{today_str}.csv"
clean_filename = f"data_cleaned_{today_str}.csv"


df_cleaned = pd.read_csv('/home/ji/NBA_Project/data/data_cleaned_03_03_2025.csv')
df_raw = pd.read_csv('/home/ji/NBA_Project/data/data_03_03_2025.csv')
                     
df = df_cleaned.merge(df_raw, on="GAME_ID", how="left")


In [152]:
df['HOME_AWAY'] = df['MATCHUP'].apply(lambda x: 'A' if '@' in x else 'H')

In [153]:
df_home = df[df['HOME_AWAY']=='H']
df_away = df[df['HOME_AWAY']=='A']

In [154]:
df_home['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df_home = df_home.sort_values(['TEAM_ID','GAME_DATE'])

/tmp/ipykernel_84724/3662264731.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_home['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])


In [155]:
 # List of stat columns for which to compute last 5 games average
stat_columns = [
        "PTS", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
        "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST",
        "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
    ]

In [156]:
 # For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5OnlyHome"
    df_home[new_col] = (df_home.groupby("TEAM_ID")[col_name]
                     .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
                     .reset_index(level=0, drop=True))



In [157]:
df_home = df_home.dropna()

In [158]:
df_away['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df_away = df_away.sort_values(['TEAM_ID','GAME_DATE'])

/tmp/ipykernel_84724/3344068307.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_away['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])


In [159]:
 # For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5Onlyaway"
    df_away[new_col] = (df_away.groupby("TEAM_ID")[col_name]
                     .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
                     .reset_index(level=0, drop=True))



In [160]:
df_away = df_away.dropna()

In [161]:
df_joined = df_away.merge(
    df_home, 
    left_on=["GAME_ID"], 
    right_on=["GAME_ID"]
    #,suffixes=('_away', '_home')
)

In [12]:
# Assuming df is your DataFrame
#df_joined.to_csv('/home/ji/NBA_Project/data/feature_engineering_home_away_stats.csv', index=False)


In [162]:
#df_joined[df_joined['GAME_DATE']=='2025-03-02']
# Get the list of column names
column_list = df_joined.columns.tolist()

print(column_list)

['PTS_LAST5_away_x', 'FGM_LAST5_away_x', 'FGA_LAST5_away_x', 'FG_PCT_LAST5_away_x', 'FG3M_LAST5_away_x', 'FG3A_LAST5_away_x', 'FG3_PCT_LAST5_away_x', 'FTM_LAST5_away_x', 'FTA_LAST5_away_x', 'FT_PCT_LAST5_away_x', 'OREB_LAST5_away_x', 'DREB_LAST5_away_x', 'REB_LAST5_away_x', 'AST_LAST5_away_x', 'STL_LAST5_away_x', 'BLK_LAST5_away_x', 'TOV_LAST5_away_x', 'PF_LAST5_away_x', 'PLUS_MINUS_LAST5_away_x', 'PTS_LAST5_home_x', 'FGM_LAST5_home_x', 'FGA_LAST5_home_x', 'FG_PCT_LAST5_home_x', 'FG3M_LAST5_home_x', 'FG3A_LAST5_home_x', 'FG3_PCT_LAST5_home_x', 'FTM_LAST5_home_x', 'FTA_LAST5_home_x', 'FT_PCT_LAST5_home_x', 'OREB_LAST5_home_x', 'DREB_LAST5_home_x', 'REB_LAST5_home_x', 'AST_LAST5_home_x', 'STL_LAST5_home_x', 'BLK_LAST5_home_x', 'TOV_LAST5_home_x', 'PF_LAST5_home_x', 'PLUS_MINUS_LAST5_home_x', 'WL_away_x', 'GAME_ID', 'SEASON_ID_x', 'TEAM_ID_x', 'TEAM_ABBREVIATION_x', 'TEAM_NAME_x', 'GAME_DATE_x', 'MATCHUP_x', 'WL_x', 'MIN_x', 'PTS_x', 'FGM_x', 'FGA_x', 'FG_PCT_x', 'FG3M_x', 'FG3A_x', 'FG3_

In [125]:
df[['TEAM_NAME_away','TEAM_NAME_home',
    'PTS_away','PTS_home','PTS_LAST5_away_away','PTS_LAST5_home_home','PTS_LAST5Onlyaway','PTS_LAST5OnlyHome']]

,TEAM_NAME_away,TEAM_NAME_home,PTS_away,PTS_home,PTS_LAST5_away_away,PTS_LAST5_home_home,PTS_LAST5Onlyaway,PTS_LAST5OnlyHome
1807,New Orleans Pelicans,Utah Jazz,128,121,111.6,112.2,110.8,112.2
2252,Chicago Bulls,Indiana Pacers,112,127,122.4,120.6,120.4,119.6
3155,Denver Nuggets,Boston Celtics,103,110,120.0,113.2,120.0,118.4
4514,LA Clippers,Los Angeles Lakers,102,108,108.4,111.4,108.4,110.6
6352,Minnesota Timberwolves,Phoenix Suns,116,98,117.4,123.8,114.2,118.6
7257,New York Knicks,Miami Heat,116,112,109.4,115.0,114.6,114.4
9534,Portland Trail Blazers,Cleveland Cavaliers,129,133,121.4,125.2,120.4,129.6
10904,Oklahoma City Thunder,San Antonio Spurs,146,132,130.4,109.0,125.0,118.6
11352,Toronto Raptors,Orlando Magic,104,102,109.0,105.0,101.6,102.6


In [166]:
 # List of stat columns for which to compute last 5 games average
stat_columns = [
        "PTS", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
        "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST",
        "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
    ]
columns =[]
 # For each column, compute the rolling average of the last 5 games (excluding the current game) 
for col_name in stat_columns:
    new_col = col_name + "_LAST5OnlyHome"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5Onlyaway"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5_away_x"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5_home_x"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5_away_y"
   # print(new_col)
    columns.append(new_col)
    new_col = col_name + "_LAST5_home_y"
   # print(new_col)
    columns.append(new_col)
columns.append('WL_away_y')
print(columns)
#df[df[columns]].head()

['PTS_LAST5OnlyHome', 'PTS_LAST5Onlyaway', 'PTS_LAST5_away_x', 'PTS_LAST5_home_x', 'PTS_LAST5_away_y', 'PTS_LAST5_home_y', 'FGM_LAST5OnlyHome', 'FGM_LAST5Onlyaway', 'FGM_LAST5_away_x', 'FGM_LAST5_home_x', 'FGM_LAST5_away_y', 'FGM_LAST5_home_y', 'FGA_LAST5OnlyHome', 'FGA_LAST5Onlyaway', 'FGA_LAST5_away_x', 'FGA_LAST5_home_x', 'FGA_LAST5_away_y', 'FGA_LAST5_home_y', 'FG_PCT_LAST5OnlyHome', 'FG_PCT_LAST5Onlyaway', 'FG_PCT_LAST5_away_x', 'FG_PCT_LAST5_home_x', 'FG_PCT_LAST5_away_y', 'FG_PCT_LAST5_home_y', 'FG3M_LAST5OnlyHome', 'FG3M_LAST5Onlyaway', 'FG3M_LAST5_away_x', 'FG3M_LAST5_home_x', 'FG3M_LAST5_away_y', 'FG3M_LAST5_home_y', 'FG3A_LAST5OnlyHome', 'FG3A_LAST5Onlyaway', 'FG3A_LAST5_away_x', 'FG3A_LAST5_home_x', 'FG3A_LAST5_away_y', 'FG3A_LAST5_home_y', 'FG3_PCT_LAST5OnlyHome', 'FG3_PCT_LAST5Onlyaway', 'FG3_PCT_LAST5_away_x', 'FG3_PCT_LAST5_home_x', 'FG3_PCT_LAST5_away_y', 'FG3_PCT_LAST5_home_y', 'FTM_LAST5OnlyHome', 'FTM_LAST5Onlyaway', 'FTM_LAST5_away_x', 'FTM_LAST5_home_x', 'FTM_LAST

In [167]:
df_joined[columns]

,PTS_LAST5OnlyHome,PTS_LAST5Onlyaway,PTS_LAST5_away_x,PTS_LAST5_home_x,PTS_LAST5_away_y,PTS_LAST5_home_y,FGM_LAST5OnlyHome,FGM_LAST5Onlyaway,FGM_LAST5_away_x,FGM_LAST5_home_x,...,PF_LAST5_home_x,PF_LAST5_away_y,PF_LAST5_home_y,PLUS_MINUS_LAST5OnlyHome,PLUS_MINUS_LAST5Onlyaway,PLUS_MINUS_LAST5_away_x,PLUS_MINUS_LAST5_home_x,PLUS_MINUS_LAST5_away_y,PLUS_MINUS_LAST5_home_y,WL_away_y
0,101.000000,102.000000,107.6,98.0,107.6,98.0,37.000000,40.000000,37.8,35.0,...,24.0,23.0,24.0,1.000000,-7.0,2.4,-3.2,2.4,-3.2,0
1,91.000000,97.000000,106.0,91.4,106.0,91.4,34.000000,39.000000,38.2,34.4,...,18.2,23.2,18.2,2.333333,-4.5,2.8,-1.0,2.8,-1.0,0
2,86.333333,104.333333,103.6,93.8,103.6,93.8,33.333333,40.333333,37.8,37.0,...,25.0,23.8,25.0,-12.000000,-4.0,1.0,-6.6,1.0,-6.6,1
3,104.000000,101.000000,105.4,106.4,105.4,106.4,36.000000,37.000000,36.8,36.8,...,17.8,19.6,17.8,1.000000,-1.5,4.8,-0.8,4.8,-0.8,0
4,98.800000,99.600000,103.2,101.8,103.2,101.8,38.800000,37.600000,40.0,40.6,...,22.0,15.8,22.0,4.200000,-7.8,-2.8,6.0,-2.8,6.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13609,131.800000,99.400000,98.8,131.2,98.8,131.2,51.200000,34.400000,35.6,51.0,...,19.8,16.2,19.8,19.800000,-8.6,-6.0,20.8,-6.0,20.8,0
13610,114.000000,98.400000,98.4,109.2,98.4,109.2,40.800000,34.800000,34.8,39.4,...,21.0,16.0,21.0,8.400000,-9.0,-9.0,-11.6,-9.0,-11.6,0
13611,117.600000,95.600000,95.6,122.4,95.6,122.4,42.000000,34.800000,34.8,45.2,...,19.0,16.6,19.0,-1.800000,-17.6,-17.6,-3.4,-17.6,-3.4,0
13612,113.000000,95.400000,95.4,119.0,95.4,119.0,38.600000,34.400000,34.4,42.8,...,18.6,17.8,18.6,4.400000,-24.4,-24.4,13.0,-24.4,13.0,0
